# Power Analysis via Simulation -- Attestation Trust Study  (PYTHON / Jupyter)

3 (attestation) x 2 (correctness) x 2 (stakes) within-subjects design

Crossed random effects: participant x item (stem)
Outcome: state trust, 7-point (treated as continuous for the LMM)

 WHAT THIS DOES (same logic as the simr version, hand-rolled in Python):
   1. Simulate a dataset under effect sizes YOU assume.
   2. Fit a mixed model.
   3. Check whether the target effect is significant.
   4. Repeat many times -> proportion significant = POWER.
   5. Sweep N until power for your target effect ~= 0.80.

 HONEST CAVEAT: statsmodels MixedLM handles CROSSED random effects (participant x item) less cleanly than R's lme4/simr. This script models the participant random intercept as the primary grouping and adds the item random intercept via variance components (vc_formula). If you hit convergence issues, the fallback is participant-only random intercepts (slightly anti-conservative) or running the R/simr version in DataSpell's R kernel. For most pilots this Python version is adequate; for the FINAL reported number, cross-check in simr.

 Runs in a plain Jupyter notebook: pip install numpy pandas statsmodels scipy

In [2]:
"""Power analysis via Monte Carlo simulation for the Attestation Trust Study.

The study uses a 3 (attestation) x 2 (correctness) x 2 (stakes) within-subjects
factorial design with crossed random effects (participant x item). The outcome
is a 7-point state-trust rating, treated as continuous for the linear
mixed-effects model.

The simulation estimates statistical power by:

1. Generating a dataset under a set of assumed fixed-effect sizes and
   random-effect standard deviations.
2. Fitting a mixed-effects model to that dataset.
3. Recording whether each target effect is statistically significant.
4. Repeating many times; the proportion of significant results is the power.
5. Sweeping the participant count until the target effect reaches ~0.80 power.

The output is only as reliable as the assumed effect sizes in ``BETA``. Estimate
those from pilot data, then re-run to obtain a defensible recruitment target.

Note:
    ``statsmodels`` handles crossed random effects less cleanly than R's
    ``lme4``/``simr``. This script models the participant random intercept as
    the primary grouping and adds the item random intercept as a variance
    component. For the final reported sample size, cross-check the number in the
    companion ``simr`` script.

Requires: numpy, pandas, statsmodels, scipy.
"""

from __future__ import annotations

import warnings
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")  # silence MixedLM convergence chatter

RNG: np.random.Generator = np.random.default_rng(6795)  # reproducible


# =============================================================================
# PARAMETERS -- the knobs. Edit these, ideally with pilot-derived values.
# =============================================================================

#: Number of distinct question stems (6 low-stakes + 6 high-stakes).
N_STEMS: int = 12

#: Assumed fixed-effect sizes, in points on the 7-point trust scale.
#: Replace with pilot-derived estimates before trusting the output.
BETA: Dict[str, float] = dict(
    intercept=4.0,        # grand-mean trust
    attestation=0.40,     # per attestation level (coded 0, 1, 2) -- H1
    correctness=0.30,     # correct - incorrect
    stakes=0.10,          # high - low
    att_x_corr=0.30,      # attestation x correctness -- H2 (core)
    att_x_stakes=0.10,    # attestation x stakes
    corr_x_stakes=0.10,   # correctness x stakes
    three_way=0.15,       # attestation x correctness x stakes -- H4
)

#: Between-participant SD of the random intercept (baseline-trust variability).
SD_PARTICIPANT: float = 0.80
#: Between-stem SD of the random intercept.
SD_ITEM: float = 0.30
#: Trial-level residual SD.
SD_RESIDUAL: float = 1.10

#: Number of simulations per sample size (200 to explore; 500-1000 to report).
N_SIMS: int = 200
#: Participant counts to evaluate.
N_TO_TEST: List[int] = [80, 120, 160, 200, 240, 300]
#: Power threshold of interest.
TARGET_POWER: float = 0.80
#: Significance threshold.
ALPHA: float = 0.05

#: Effects to evaluate power for, mapping a display label to the model term.
#: The core effect is the H2 two-way; the H4 three-way is also evaluated.
TARGET_TERMS: Dict[str, str] = {
    "H2_attestation:correctness": "attestation:correctness",
    "H4_attestation:correctness:stakes": "attestation:correctness:stakes",
}


# =============================================================================
# SIMULATION
# =============================================================================
def simulate_dataset(
    n_participants: int,
    betas: Dict[str, float] = BETA,
    sd_part: float = SD_PARTICIPANT,
    sd_item: float = SD_ITEM,
    sd_resid: float = SD_RESIDUAL,
    n_stems: int = N_STEMS,
    rng: np.random.Generator = RNG,
) -> pd.DataFrame:
    """Generate one simulated dataset for the 3x2x2 within-subjects design.

    Each participant contributes one trial per cell (12 trials total). Stems are
    rotated across cells by participant so that item is crossed with condition,
    mirroring the study's Latin-square counterbalancing. The outcome is built
    from the assumed fixed effects plus participant and item random intercepts
    and trial-level residual noise.

    Args:
        n_participants: Number of simulated participants.
        betas: Fixed-effect sizes keyed by term name (see :data:`BETA`).
        sd_part: SD of the participant random intercept.
        sd_item: SD of the item (stem) random intercept.
        sd_resid: SD of the trial-level residual noise.
        n_stems: Number of distinct stems to rotate across cells.
        rng: NumPy random generator, supplied for reproducibility.

    Returns:
        A long-format DataFrame with one row per participant-trial and columns
        ``participant``, ``item`` (both categorical), ``attestation`` (0/1/2),
        ``correctness`` (-0.5/0.5), ``stakes`` (-0.5/0.5), and ``y`` (the
        simulated trust rating).
    """
    att_levels = [0, 1, 2]        # None / Weak / Strong (linear-trend coding)
    corr_levels = [-0.5, 0.5]     # incorrect / correct (centered)
    stake_levels = [-0.5, 0.5]    # low / high (centered)
    cells: List[Tuple[int, float, float]] = [
        (a, c, s) for a in att_levels for c in corr_levels for s in stake_levels
    ]

    part_re = rng.normal(0, sd_part, size=n_participants)
    item_re = rng.normal(0, sd_item, size=n_stems)

    rows: List[Tuple[int, int, int, float, float, float]] = []
    for p in range(n_participants):
        for i, (a, c, s) in enumerate(cells):
            # rotate stem assignment by participant so item is crossed with cell
            item = (i + p) % n_stems
            mu = (
                betas["intercept"]
                + betas["attestation"] * a
                + betas["correctness"] * c
                + betas["stakes"] * s
                + betas["att_x_corr"] * a * c
                + betas["att_x_stakes"] * a * s
                + betas["corr_x_stakes"] * c * s
                + betas["three_way"] * a * c * s
                + part_re[p]
                + item_re[item]
            )
            y = mu + rng.normal(0, sd_resid)
            rows.append((p, item, a, c, s, y))

    df = pd.DataFrame(
        rows,
        columns=["participant", "item", "attestation", "correctness", "stakes", "y"],
    )
    df["participant"] = df["participant"].astype("category")
    df["item"] = df["item"].astype("category")
    return df


# =============================================================================
# MODEL FITTING
# =============================================================================
def fit_and_test(df: pd.DataFrame, term: str) -> float:
    """Fit the mixed-effects model and return the p-value for one term.

    Fits ``y ~ attestation * correctness * stakes`` with a participant random
    intercept and an item variance component (the crossed random-effects
    structure). If the requested term is not found by its exact label, the
    function falls back to matching by the set of interacting factors.

    Args:
        df: A dataset produced by :func:`simulate_dataset`.
        term: The model term to test, e.g. ``"attestation:correctness"``.

    Returns:
        The two-sided p-value for ``term``, or ``float('nan')`` if the model
        failed to converge or the term could not be located.
    """
    vc = {"item": "0 + C(item)"}  # item variance component, crossed structure
    model = smf.mixedlm(
        "y ~ attestation * correctness * stakes",
        data=df,
        groups=df["participant"],
        vc_formula=vc,
        re_formula="1",  # participant random intercept
    )
    try:
        res = model.fit(reml=False, method="lbfgs", maxiter=200)
    except Exception:
        return float("nan")

    if term not in res.pvalues.index:
        # statsmodels may name interaction terms differently; match by factor set
        candidates = [
            ix for ix in res.pvalues.index if set(ix.split(":")) == set(term.split(":"))
        ]
        if not candidates:
            return float("nan")
        term = candidates[0]
    return float(res.pvalues[term])


# =============================================================================
# POWER ESTIMATION
# =============================================================================
def power_at_N(
    n_participants: int,
    target_terms: Dict[str, str] = TARGET_TERMS,
    n_sims: int = N_SIMS,
    alpha: float = ALPHA,
) -> Tuple[Dict[str, float], Dict[str, int]]:
    """Estimate power for each target term at a single sample size.

    Simulates ``n_sims`` datasets at the given participant count, fits the model
    to each, and records the fraction of fits in which each target term is
    significant at ``alpha``. Power is computed only over fits that converged.

    Args:
        n_participants: Number of participants to simulate per dataset.
        target_terms: Mapping of display label to model term (see
            :data:`TARGET_TERMS`).
        n_sims: Number of simulated datasets.
        alpha: Significance threshold.

    Returns:
        A tuple ``(power, valid)`` where ``power`` maps each label to its
        estimated power (significant fits / converged fits) and ``valid`` maps
        each label to the count of converged fits (a diagnostic; many failures
        indicate the crossed structure is straining ``statsmodels``).
    """
    hits: Dict[str, int] = {name: 0 for name in target_terms}
    valid: Dict[str, int] = {name: 0 for name in target_terms}

    for _ in range(n_sims):
        df = simulate_dataset(n_participants)
        for name, term in target_terms.items():
            p = fit_and_test(df, term)
            if not np.isnan(p):
                valid[name] += 1
                if p < alpha:
                    hits[name] += 1

    power: Dict[str, float] = {}
    for name in target_terms:
        v = valid[name]
        power[name] = hits[name] / v if v else float("nan")
    return power, valid


def power_curve(n_list: List[int] = N_TO_TEST) -> pd.DataFrame:
    """Estimate and print power across a range of sample sizes.

    For each sample size, estimates power for every term in
    :data:`TARGET_TERMS`, prints a row of the resulting power curve, and flags
    any cell that meets :data:`TARGET_POWER`.

    Args:
        n_list: Participant counts to evaluate.

    Returns:
        A DataFrame with one row per sample size; column ``N`` holds the count
        and one column per target label holds the estimated power.
    """
    print(f"Power simulation: {N_SIMS} sims per N, alpha={ALPHA}\n")
    print(f"{'N':>5} | " + " | ".join(f"{name:>38}" for name in TARGET_TERMS))
    print("-" * (8 + 41 * len(TARGET_TERMS)))

    results: List[Dict[str, float]] = []
    for n in n_list:
        power, valid = power_at_N(n)
        row: Dict[str, float] = {"N": n}
        cells: List[str] = []
        for name in TARGET_TERMS:
            pw = power[name]
            row[name] = pw
            flag = "  <-- >=80%" if (not np.isnan(pw) and pw >= TARGET_POWER) else ""
            cells.append(f"{pw:>.2f} (valid {valid[name]}/{N_SIMS}){flag}")
        print(f"{n:>5} | " + " | ".join(f"{c:>38}" for c in cells))
        results.append(row)

    return pd.DataFrame(results)


def main() -> pd.DataFrame:
    """Run the full power curve and print reading guidance.

    Returns:
        The power-curve DataFrame produced by :func:`power_curve`.
    """
    print("=" * 78)
    print("ATTESTATION TRUST STUDY -- power simulation (Python/statsmodels)")
    print("Effect sizes are ASSUMPTIONS. Replace BETA with pilot estimates,")
    print("then trust the N where your target effect reaches 0.80.")
    print("=" * 78, "\n")

    df_results = power_curve()

    print("\n" + "=" * 78)
    print("HOW TO READ THIS")
    print("=" * 78)
    print(
        """
- Find the smallest N where H2 (attestation:correctness, your CORE effect)
  reaches ~0.80. That is your minimum defensible N for the primary hypothesis.
- Check H4 (three-way) at that N. If also ~0.80, you can power for H4 too.
  If it lags (likely), either raise N to where H4 hits 0.80 (budget permitting)
  or keep the H2 target and report H4 as exploratory.
- 'valid' shows how many sims converged; if many failed, MixedLM is struggling
  with the crossed structure -- cross-check that N in the R/simr version.
- The numbers are only as good as the BETA assumptions. Pilot first, then
  re-run with pilot-derived effect sizes and SDs. Recruit the resulting N,
  not more (avoid over-recruiting).
"""
    )
    return df_results


if __name__ == "__main__":
    main()

ATTESTATION TRUST STUDY -- power simulation (Python/statsmodels)
Effect sizes are ASSUMPTIONS. Replace BETA with pilot estimates,
then trust the N where your target effect reaches 0.80.

Power simulation: 200 sims per N, alpha=0.05

    N |             H2_attestation:correctness |      H4_attestation:correctness:stakes
------------------------------------------------------------------------------------------
   80 |        0.91 (valid 200/200)  <-- >=80% |                   0.13 (valid 200/200)
  120 |        0.98 (valid 200/200)  <-- >=80% |                   0.17 (valid 200/200)
  160 |        1.00 (valid 200/200)  <-- >=80% |                   0.24 (valid 200/200)
  200 |        1.00 (valid 200/200)  <-- >=80% |                   0.25 (valid 200/200)
  240 |        1.00 (valid 200/200)  <-- >=80% |                   0.28 (valid 200/200)
  300 |        1.00 (valid 200/200)  <-- >=80% |                   0.39 (valid 200/200)

HOW TO READ THIS

- Find the smallest N where H2 (attestati